# Perbandingan eksperimen GraphSAGE

Notebook ini membaca **seluruh run EXP1–EXP11 secara otomatis** dari `result/`, termasuk format JSON lama dan folder run baru. Tidak membutuhkan dataset, checkpoint, GPU, atau training ulang.

Hasil: grafik antar eksperimen, grafik setiap run per eksperimen/metode, kurva training, katalog semua metrik numerik, CSV, dan **laporan HTML interaktif offline** dengan pilihan seluruh metrik utama maupun diagnostik.

**Cara pakai:** pilih kernel Python dengan dependensi `notebooks/requirements.txt`, lalu **Run All**. Di VS Code lokal, environment yang disiapkan adalah `notebooks/.venv/Scripts/python.exe`. Setelah run baru ditambahkan, jalankan ulang notebook.

In [ ]:
# Jalankan hanya jika dependensi belum terpasang pada kernel yang dipilih:
# %pip install -r requirements.txt  # jika working directory adalah notebooks/
# %pip install -r notebooks/requirements.txt  # jika working directory adalah root repository

## Konfigurasi

`None` berarti semua nilai. Run main dipakai untuk grafik awal; smoke dan pilot dapat diaktifkan terpisah. Laporan HTML tetap memuat seluruh run yang dibaca, sehingga semua hasil bisa ditelusuri tanpa mengubah notebook.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import plotly.io as pio
from IPython.display import display, Markdown, FileLink

# Bisa dijalankan dari root repository maupun folder notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'result').is_dir() and (p / 'notebooks' / 'compare_results.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Buka notebook dari repository ini atau isi ROOT dengan path repository.')
sys.path.insert(0, str(ROOT / 'notebooks'))
from compare_results import (load_results, plot_comparison, plot_runs, plot_history,
                             summary_table, export_report, natural_key, MAIN_METRICS)

RESULT_DIR = ROOT / 'result'
OUTPUT_DIR = ROOT / 'notebooks' / 'output'
EXPERIMENTS = None             # Contoh: ['exp9', 'exp10', 'exp11']
METHODS = None                 # Contoh: ['uniform', 'topology', 'importance']
KINDS = ['main']               # Tambahkan 'pilot' dan/atau 'smoke' bila diperlukan.
COMPARISON_IDS = None          # Contoh: ['533260b8f51244e6']
METRICS = MAIN_METRICS         # Isi None untuk grafik SEMUA metrik utama.
FOCUS_EXPERIMENT = 'exp11'
RUN_METRIC = 'auprc'           # Bisa diganti dengan metrik numerik apa pun dari katalog.
HISTORY_METRICS = ['loss', 'val_auprc']  # Isi None untuk semua metrik history.

# Renderer yang didukung notebook VS Code / JupyterLab, tanpa URL CDN.
pio.renderers.default = 'plotly_mimetype'
pd.set_option('display.max_colwidth', 100)

## Inventaris dan kualitas data

Satu file hasil adalah satu run. `summary.csv` sumber tidak dimuat sebagai run tambahan. Identitas `comparison_id`, seed, dan attempt dipertahankan. Run yang belum `complete` tidak masuk analisis metrik akhir. Format lama tanpa status ditandai `metrics_available`.

In [ ]:
data = load_results(RESULT_DIR)
runs = data.select(experiments=EXPERIMENTS, methods=METHODS, kinds=KINDS,
                   comparison_ids=COMPARISON_IDS)
if runs.empty:
    raise ValueError('Tidak ada run sesuai filter. Periksa konfigurasi di atas.')
print(f'{len(data.runs)} run total; {len(runs)} run terpilih; '
      f'{data.metrics.metric.nunique()} metrik akhir/diagnostik; '
      f'{data.history.metric.nunique()} metrik history.')
display(runs.groupby(['experiment', 'kind', 'comparison_id', 'strategy'], sort=False)
        .agg(runs=('run_uid', 'size'), seeds=('seed', 'nunique'),
             test_count=('test_count', 'first')).reset_index())
if not data.issues.empty:
    display(Markdown('**File yang dilewati / catatan pembacaan:**'))
    display(data.issues)

## Katalog seluruh metrik

`primary` = angka dalam objek `metrics`. `diagnostic` = angka lain seperti metrik threshold alternatif, channel pembayaran, temporal bins, timing, statistik prediksi, dan diagnostik training. Array diratakan dengan indeks `[0]`, `[1]`, dst. String/boolean menjadi metadata, bukan skor. Nilai null tetap kosong. Metrik yang belum ada pada eksperimen lama **tidak diisi nol atau dihitung ulang**.

In [ ]:
catalog = data.catalog()
display(catalog)
print('Metrik history:', sorted(data.history.metric.unique(), key=natural_key))
# Contoh pencarian:
display(catalog[catalog.metric.str.contains('f1|recall|auprc|inference', regex=True)].head(40))

## Perbandingan antar eksperimen dan metode

Grafik awal menampilkan metrik utama yang umum dipakai. Ubah `METRICS = None` untuk seluruh metrik utama, atau pilih metrik apa pun melalui laporan HTML di akhir notebook.

Mean ± standar deviasi sampel dihitung **per comparison_id dan metode**, memakai attempt terbaru untuk tiap seed. Titik menunjukkan nilai seed. Jika hanya ada satu seed, SD tidak ditampilkan. Perbandingan antar eksperimen bersifat deskriptif karena data dan protokol berubah; jumlah data test dicantumkan pada label.

In [ ]:
metric_names = (catalog.loc[catalog.section == 'primary', 'metric'].tolist()
                if METRICS is None else METRICS)
for metric in metric_names:
    display(plot_comparison(data, metric, runs))

## Setiap run pada masing-masing eksperimen

Satu batang = satu seed/attempt. Semua attempt tetap ditampilkan di sini. Ubah `RUN_METRIC` untuk menampilkan metrik lain. Jika eksperimen memiliki beberapa konfigurasi, ID-nya ditampilkan pada label.

In [ ]:
for experiment in sorted(runs.experiment.unique(), key=natural_key):
    display(Markdown(f'### {experiment.upper()} — {RUN_METRIC}'))
    display(plot_runs(data, RUN_METRIC, runs[runs.experiment == experiment]))

## Setiap run dari masing-masing metode

Bagian ini memperlihatkan seed dan attempt untuk tiap metode pada eksperimen fokus. Ubah `FOCUS_EXPERIMENT` atau filter `COMPARISON_IDS` pada sel konfigurasi. Laporan HTML menyediakan pilihan yang sama tanpa menjalankan ulang sel.

In [ ]:
focus = runs[runs.experiment == FOCUS_EXPERIMENT]
if focus.empty:
    print(f'{FOCUS_EXPERIMENT} tidak ada dalam filter. Ubah FOCUS_EXPERIMENT untuk menampilkan detail.')
else:
    for method in focus.strategy.unique():
        display(Markdown(f'### {FOCUS_EXPERIMENT.upper()} / {method}'))
        display(plot_runs(data, RUN_METRIC, focus[focus.strategy == method]))

## Kurva training setiap run

Garis berhenti pada epoch terakhir run; tidak ada pengisian nilai setelah early stopping. Bintang menandai `best_epoch` yang tersimpan. Metrik nested pada history, misalnya `validation_blocks[0].ap_lift`, juga dapat dipilih.

In [ ]:
history_names = (sorted(data.history.metric.unique(), key=natural_key)
                 if HISTORY_METRICS is None else HISTORY_METRICS)
for metric in history_names:
    display(plot_history(data, metric, focus))
# Untuk eksperimen lain: plot_history(data, 'loss', runs[runs.experiment == 'exp10'])

## Metrik tambahan dan tabel ringkasan

Path membedakan metrik utama, fixed threshold 0.5, validation threshold, dan subgroup. Jangan menggabungkan `selection_score` yang memakai `selection_metric` berbeda. Untuk bin temporal, indeks sama tidak menjamin rentang tanggal yang sama.

In [ ]:
DIAGNOSTIC_METRIC = 'test_metrics_at_fixed_threshold_0_5.f1'
if DIAGNOSTIC_METRIC in set(data.metrics.metric):
    display(plot_runs(data, DIAGNOSTIC_METRIC, focus))
summary = summary_table(data, runs)
display(summary[(summary.section == 'primary') & summary.metric.isin(MAIN_METRICS)])

## Ekspor laporan lengkap

HTML ini berisi **semua metrik** dan **semua run** yang berhasil dibaca, dengan tiga tampilan: antar eksperimen, setiap run, dan riwayat training. Filter awal hanya menampilkan main; centang pilot/smoke untuk melihatnya. Pilih satu metode untuk membandingkan semua run metode tersebut. Tombol sebelumnya/berikutnya menelusuri seluruh metrik.

Buka HTML di browser biasa; file mandiri ini tidak memerlukan internet. Grafik bisa diunduh sebagai PNG melalui toolbar atau SVG melalui tombol. CSV menyimpan inventaris run, metrik lengkap, history, ringkasan count/mean/std, katalog, dan catatan pembacaan.

In [ ]:
report_path = export_report(data, OUTPUT_DIR)
print('Buka file ini di browser:', report_path)
display(FileLink(str(report_path)))
print('CSV tersimpan di:', OUTPUT_DIR)